# Build matched Layer-1 MLP delta vectors for low-rank analysis

In [2]:
from google.colab import drive
from pathlib import Path
import os
import shutil


MOUNTPOINT = Path("/content/drive")


# Safely disconnect an existing Drive mount, if one exists.
try:
    drive.flush_and_unmount()
    print("Existing Drive mount disconnected.")
except Exception as error:
    print(
        "No active Drive mount to disconnect:",
        type(error).__name__,
    )


# Remove stale local placeholder files/directories.
if MOUNTPOINT.exists():
    print(
        "Removing stale mountpoint contents:",
        list(MOUNTPOINT.iterdir()),
    )

    shutil.rmtree(
        MOUNTPOINT,
        ignore_errors=True,
    )

MOUNTPOINT.mkdir(
    parents=True,
    exist_ok=True,
)

assert not any(MOUNTPOINT.iterdir())

print("Clean mountpoint prepared.")


# Mount the actual Google Drive.
drive.mount(
    str(MOUNTPOINT),
    force_remount=False,
)


BASE_DIR = Path(
    "/content/drive/MyDrive/algoverse/dnabert"
)

OUTPUT_DIR = BASE_DIR / "outputs"

NOTEBOOK6_DIR = (
    OUTPUT_DIR
    / "gc_controlled_interventions"
)

NOTEBOOK7_DIR = (
    OUTPUT_DIR
    / "promoter_mlp_low_rank_subspace"
)

print("\nNotebook 6 directory:")
print(NOTEBOOK6_DIR)

print(
    "Exists:",
    NOTEBOOK6_DIR.exists(),
)


notebook6_files = (
    sorted(NOTEBOOK6_DIR.glob("*"))
    if NOTEBOOK6_DIR.exists()
    else []
)

print("\nFiles visible in Notebook 6 directory:")

for path in notebook6_files:
    print(
        path.name,
        f"({path.stat().st_size / 1024:.1f} KB)",
    )

print(
    "\nNumber of Notebook 6 files:",
    len(notebook6_files),
)


required_filenames = [
    "gc_intervention_scored_results.csv",
    "gc_unique_sequence_layer1_mlp_index.csv",
    "gc_unique_sequence_layer1_mlp_representations.npz",
    "layer1_mlp_promoter_direction.npz",
]

print("\nRequired Notebook 7 inputs:")

for filename in required_filenames:
    required_path = (
        NOTEBOOK6_DIR / filename
    )

    print(
        filename,
        "FOUND"
        if required_path.exists()
        else "MISSING",
    )

Drive not mounted, so nothing to flush and unmount.
Existing Drive mount disconnected.
Clean mountpoint prepared.
Mounted at /content/drive

Notebook 6 directory:
/content/drive/MyDrive/algoverse/dnabert/outputs/gc_controlled_interventions
Exists: True

Files visible in Notebook 6 directory:
NOTEBOOK6_RESULTS.md (1.2 KB)
gc_dose_response_summary.csv (0.9 KB)
gc_intervention_cross_validated_direction_bootstrap.csv (2.0 KB)
gc_intervention_cross_validated_direction_scores.csv (6025.3 KB)
gc_intervention_generation_summary.csv (0.6 KB)
gc_intervention_initial_score_summary.csv (2.4 KB)
gc_intervention_mlp_direction_contrasts.csv (318.6 KB)
gc_intervention_mlp_direction_scores.csv (5745.3 KB)
gc_intervention_mlp_direction_summary.csv (0.7 KB)
gc_intervention_scored_results.csv (5194.2 KB)
gc_intervention_variants.csv (3581.6 KB)
gc_matched_neutral_contrasts.csv (488.3 KB)
gc_source_cluster_bootstrap_summary.csv (2.0 KB)
gc_source_level_contrasts.csv (54.1 KB)
gc_unique_sequence_layer1_mlp_

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/algoverse/dnabert"
)

OUTPUT_DIR = BASE_DIR / "outputs"

NOTEBOOK6_DIR = (
    OUTPUT_DIR
    / "gc_controlled_interventions"
)

NOTEBOOK7_DIR = (
    OUTPUT_DIR
    / "promoter_mlp_low_rank_subspace"
)

NOTEBOOK7_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# Reload Notebook 6 outputs
# ------------------------------------------------------------

GC_SCORED_PATH = (
    NOTEBOOK6_DIR
    / "gc_intervention_scored_results.csv"
)

LAYER1_INDEX_PATH = (
    NOTEBOOK6_DIR
    / "gc_unique_sequence_layer1_mlp_index.csv"
)

LAYER1_REPRESENTATION_PATH = (
    NOTEBOOK6_DIR
    / "gc_unique_sequence_layer1_mlp_representations.npz"
)

PROMOTER_DIRECTION_PATH = (
    NOTEBOOK6_DIR
    / "layer1_mlp_promoter_direction.npz"
)


gc_intervention_scored_df = pd.read_csv(
    GC_SCORED_PATH
)

layer1_mlp_sequence_index_df = pd.read_csv(
    LAYER1_INDEX_PATH
)

layer1_representation_archive = np.load(
    LAYER1_REPRESENTATION_PATH
)

unique_layer1_mlp_representations = (
    layer1_representation_archive[
        "representations"
    ]
)

promoter_direction_archive = np.load(
    PROMOTER_DIRECTION_PATH
)

promoter_direction_unit = (
    promoter_direction_archive[
        "promoter_direction_unit"
    ]
)


print(
    "Scored intervention rows:",
    len(gc_intervention_scored_df),
)

print(
    "Layer-1 representation matrix:",
    unique_layer1_mlp_representations.shape,
)

print(
    "Promoter direction shape:",
    promoter_direction_unit.shape,
)


assert len(gc_intervention_scored_df) == 15120

assert (
    unique_layer1_mlp_representations.shape
    == (11718, 768)
)

assert promoter_direction_unit.shape == (768,)

assert np.isfinite(
    unique_layer1_mlp_representations
).all()

assert np.isfinite(
    promoter_direction_unit
).all()

assert np.isclose(
    np.linalg.norm(promoter_direction_unit),
    1.0,
    atol=1e-5,
)


# ------------------------------------------------------------
# Sequence-to-representation mapping
# ------------------------------------------------------------

sequence_to_representation_index = dict(
    zip(
        layer1_mlp_sequence_index_df[
            "variant_sequence"
        ],
        layer1_mlp_sequence_index_df[
            "representation_index"
        ].astype(int),
    )
)

assert (
    len(sequence_to_representation_index)
    == 11718
)


# ------------------------------------------------------------
# Construct matched intervention blocks
# ------------------------------------------------------------

block_keys = [
    "source_class",
    "source_index",
    "mutation_level",
    "variant_seed",
]

notebook7_intervention_df = (
    gc_intervention_scored_df.copy()
)

notebook7_intervention_df[
    "representation_index"
] = (
    notebook7_intervention_df[
        "variant_sequence"
    ].map(
        sequence_to_representation_index
    )
)

assert notebook7_intervention_df[
    "representation_index"
].notna().all()

notebook7_intervention_df[
    "representation_index"
] = (
    notebook7_intervention_df[
        "representation_index"
    ].astype(int)
)


representation_index_wide_df = (
    notebook7_intervention_df
    .pivot(
        index=block_keys,
        columns="intervention_type",
        values="representation_index",
    )
    .reset_index()
)

representation_index_wide_df.columns.name = None

required_conditions = [
    "original",
    "gc_increased",
    "gc_decreased",
    "gc_neutral",
]

assert representation_index_wide_df[
    required_conditions
].notna().all().all()


def get_representations_from_index_column(
    dataframe,
    column_name,
):
    indices = dataframe[
        column_name
    ].to_numpy(dtype=int)

    return unique_layer1_mlp_representations[
        indices
    ]


original_mlp = (
    get_representations_from_index_column(
        representation_index_wide_df,
        "original",
    )
)

gc_increased_mlp = (
    get_representations_from_index_column(
        representation_index_wide_df,
        "gc_increased",
    )
)

gc_decreased_mlp = (
    get_representations_from_index_column(
        representation_index_wide_df,
        "gc_decreased",
    )
)

gc_neutral_mlp = (
    get_representations_from_index_column(
        representation_index_wide_df,
        "gc_neutral",
    )
)


print("\nMatched block count:")
print(len(representation_index_wide_df))

print("\nRepresentation matrix shapes:")
print("Original:", original_mlp.shape)
print("GC increased:", gc_increased_mlp.shape)
print("GC decreased:", gc_decreased_mlp.shape)
print("GC neutral:", gc_neutral_mlp.shape)


# ------------------------------------------------------------
# Construct matched intervention vectors
# ------------------------------------------------------------

delta_gc_increased = (
    gc_increased_mlp
    - original_mlp
)

delta_gc_decreased = (
    gc_decreased_mlp
    - original_mlp
)

delta_gc_neutral = (
    gc_neutral_mlp
    - original_mlp
)

delta_gc_increase_vs_neutral = (
    gc_increased_mlp
    - gc_neutral_mlp
)

delta_gc_decrease_vs_neutral = (
    gc_decreased_mlp
    - gc_neutral_mlp
)

delta_gc_increase_vs_decrease = (
    gc_increased_mlp
    - gc_decreased_mlp
)


delta_matrices = {
    "gc_increased_vs_original":
        delta_gc_increased,

    "gc_decreased_vs_original":
        delta_gc_decreased,

    "gc_neutral_vs_original":
        delta_gc_neutral,

    "gc_increase_vs_neutral":
        delta_gc_increase_vs_neutral,

    "gc_decrease_vs_neutral":
        delta_gc_decrease_vs_neutral,

    "gc_increase_vs_decrease":
        delta_gc_increase_vs_decrease,
}


expected_shape = (
    len(representation_index_wide_df),
    768,
)

for delta_name, delta_matrix in (
    delta_matrices.items()
):
    assert delta_matrix.shape == expected_shape
    assert np.isfinite(delta_matrix).all()


# ------------------------------------------------------------
# Diagnostic statistics
# ------------------------------------------------------------

delta_diagnostic_rows = []

for delta_name, delta_matrix in (
    delta_matrices.items()
):
    vector_norms = np.linalg.norm(
        delta_matrix,
        axis=1,
    )

    direction_projections = (
        delta_matrix
        @ promoter_direction_unit
    )

    for (
        metadata_row,
        vector_norm,
        direction_projection,
    ) in zip(
        representation_index_wide_df[
            block_keys
        ].itertuples(index=False),
        vector_norms,
        direction_projections,
    ):
        delta_diagnostic_rows.append({
            "source_class":
                metadata_row.source_class,

            "source_index":
                metadata_row.source_index,

            "mutation_level":
                metadata_row.mutation_level,

            "variant_seed":
                metadata_row.variant_seed,

            "delta_type":
                delta_name,

            "delta_norm":
                float(vector_norm),

            "promoter_direction_projection":
                float(direction_projection),

            "direction_fraction_of_norm":
                float(
                    abs(direction_projection)
                    / vector_norm
                )
                if vector_norm > 0
                else np.nan,
        })


delta_diagnostic_df = pd.DataFrame(
    delta_diagnostic_rows
)


delta_diagnostic_summary = (
    delta_diagnostic_df
    .groupby(
        [
            "source_class",
            "mutation_level",
            "delta_type",
        ]
    )
    .agg(
        number_of_vectors=(
            "delta_norm",
            "size",
        ),
        mean_delta_norm=(
            "delta_norm",
            "mean",
        ),
        median_delta_norm=(
            "delta_norm",
            "median",
        ),
        mean_direction_projection=(
            "promoter_direction_projection",
            "mean",
        ),
        median_direction_projection=(
            "promoter_direction_projection",
            "median",
        ),
        mean_direction_fraction_of_norm=(
            "direction_fraction_of_norm",
            "mean",
        ),
    )
    .reset_index()
)


print(
    "\nMatched vector diagnostic summary:"
)

display(
    delta_diagnostic_summary[
        delta_diagnostic_summary[
            "delta_type"
        ].isin(
            [
                "gc_increase_vs_neutral",
                "gc_decrease_vs_neutral",
                "gc_increase_vs_decrease",
            ]
        )
    ]
)


# ------------------------------------------------------------
# Save Notebook 7 intermediate outputs
# ------------------------------------------------------------

representation_index_wide_df.to_csv(
    NOTEBOOK7_DIR
    / "matched_intervention_representation_indices.csv",
    index=False,
)

delta_diagnostic_df.to_csv(
    NOTEBOOK7_DIR
    / "matched_delta_vector_diagnostics.csv",
    index=False,
)

delta_diagnostic_summary.to_csv(
    NOTEBOOK7_DIR
    / "matched_delta_vector_diagnostic_summary.csv",
    index=False,
)

np.savez_compressed(
    NOTEBOOK7_DIR
    / "matched_layer1_mlp_delta_matrices.npz",

    gc_increased_vs_original=(
        delta_gc_increased
    ),

    gc_decreased_vs_original=(
        delta_gc_decreased
    ),

    gc_neutral_vs_original=(
        delta_gc_neutral
    ),

    gc_increase_vs_neutral=(
        delta_gc_increase_vs_neutral
    ),

    gc_decrease_vs_neutral=(
        delta_gc_decrease_vs_neutral
    ),

    gc_increase_vs_decrease=(
        delta_gc_increase_vs_decrease
    ),
)


print(
    "\nSaved matched intervention vectors to:"
)

print(
    NOTEBOOK7_DIR
    / "matched_layer1_mlp_delta_matrices.npz"
)

print(
    "\nNotebook 7 Cell 1 completed."
)

Scored intervention rows: 15120
Layer-1 representation matrix: (11718, 768)
Promoter direction shape: (768,)

Matched block count:
3780

Representation matrix shapes:
Original: (3780, 768)
GC increased: (3780, 768)
GC decreased: (3780, 768)
GC neutral: (3780, 768)

Matched vector diagnostic summary:


,source_class,mutation_level,delta_type,number_of_vectors,mean_delta_norm,median_delta_norm,mean_direction_projection,median_direction_projection,mean_direction_fraction_of_norm
0,non-promoter,4,gc_decrease_vs_neutral,945,0.898670,0.888994,-0.119271,-0.119166,0.161200
2,non-promoter,4,gc_increase_vs_decrease,945,0.963904,0.955070,0.284755,0.277914,0.301041
3,non-promoter,4,gc_increase_vs_neutral,945,0.903233,0.895665,0.165484,0.160784,0.195802
6,non-promoter,8,gc_decrease_vs_neutral,945,1.141486,1.128301,-0.200740,-0.196861,0.196182
8,non-promoter,8,gc_increase_vs_decrease,945,1.330697,1.305279,0.577825,0.583789,0.433818
9,non-promoter,8,gc_increase_vs_neutral,945,1.176210,1.155593,0.377085,0.389594,0.322893
12,promoter,4,gc_decrease_vs_neutral,945,0.933868,0.923500,-0.190959,-0.194710,0.223559
14,promoter,4,gc_increase_vs_decrease,945,1.053297,1.038106,0.420721,0.442412,0.397553
15,promoter,4,gc_increase_vs_neutral,945,0.953996,0.946334,0.229763,0.229775,0.246301
18,promoter,8,gc_decrease_vs_neutral,945,1.180615,1.165316,-0.334204,-0.353592,0.295276



Saved matched intervention vectors to:
/content/drive/MyDrive/algoverse/dnabert/outputs/promoter_mlp_low_rank_subspace/matched_layer1_mlp_delta_matrices.npz

Notebook 7 Cell 1 completed.


Cell 2: source-averaged SVD and effective rank

In [4]:
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Average the five mutation seeds for every source and dose
# ------------------------------------------------------------

source_keys = [
    "source_class",
    "source_index",
    "mutation_level",
]

block_metadata = (
    representation_index_wide_df[
        block_keys
    ]
    .reset_index(drop=True)
)


def source_average_delta_matrix(
    delta_matrix,
):
    """
    Average the five random intervention seeds for each
    source sequence and mutation level.
    """

    delta_frame = pd.DataFrame(
        delta_matrix
    )

    delta_frame[source_keys] = (
        block_metadata[source_keys]
        .reset_index(drop=True)
    )

    averaged_frame = (
        delta_frame
        .groupby(
            source_keys,
            sort=True,
        )
        .mean()
        .reset_index()
    )

    metadata = averaged_frame[
        source_keys
    ].copy()

    matrix = averaged_frame.drop(
        columns=source_keys
    ).to_numpy(dtype=np.float64)

    return metadata, matrix


source_metadata_df, source_gc_up_vs_neutral = (
    source_average_delta_matrix(
        delta_gc_increase_vs_neutral
    )
)

_, source_gc_down_vs_neutral = (
    source_average_delta_matrix(
        delta_gc_decrease_vs_neutral
    )
)

_, source_gc_up_vs_down = (
    source_average_delta_matrix(
        delta_gc_increase_vs_decrease
    )
)


print(
    "Source-averaged metadata rows:",
    len(source_metadata_df),
)

print(
    "Source-averaged matrix shape:",
    source_gc_up_vs_down.shape,
)

assert len(source_metadata_df) == 756

assert source_gc_up_vs_down.shape == (
    756,
    768,
)

assert (
    source_metadata_df
    .groupby(
        [
            "source_class",
            "mutation_level",
        ]
    )
    .size()
    .eq(189)
    .all()
)


source_delta_matrices = {
    "gc_increase_vs_neutral":
        source_gc_up_vs_neutral,

    "gc_decrease_vs_neutral":
        source_gc_down_vs_neutral,

    "gc_increase_vs_decrease":
        source_gc_up_vs_down,
}


# ------------------------------------------------------------
# SVD helper functions
# ------------------------------------------------------------

K_VALUES = [
    1,
    2,
    4,
    8,
    16,
    32,
    64,
    128,
]


def rank_needed(
    cumulative_energy,
    threshold,
):
    return int(
        np.searchsorted(
            cumulative_energy,
            threshold,
        )
        + 1
    )


def analyze_svd(
    matrix,
    center,
):
    """
    Uncentered SVD measures total intervention-vector energy,
    including the shared mean effect.

    Centered SVD measures variation around that mean effect.
    """

    analysis_matrix = matrix.copy()

    mean_vector = analysis_matrix.mean(
        axis=0
    )

    if center:
        analysis_matrix = (
            analysis_matrix
            - mean_vector
        )

    _, singular_values, right_vectors = (
        np.linalg.svd(
            analysis_matrix,
            full_matrices=False,
        )
    )

    squared_singular_values = (
        singular_values ** 2
    )

    energy_fraction = (
        squared_singular_values
        / squared_singular_values.sum()
    )

    cumulative_energy = np.cumsum(
        energy_fraction
    )

    participation_ratio = (
        squared_singular_values.sum() ** 2
        / np.square(
            squared_singular_values
        ).sum()
    )

    return {
        "mean_vector": mean_vector,
        "singular_values": singular_values,
        "right_vectors": right_vectors,
        "energy_fraction": energy_fraction,
        "cumulative_energy": cumulative_energy,
        "participation_ratio": (
            participation_ratio
        ),
    }


# ------------------------------------------------------------
# Analyze pooled and stratified matrices
# ------------------------------------------------------------

svd_summary_rows = []
component_rows = []
svd_results = {}

analysis_groups = [
    (
        "all",
        np.ones(
            len(source_metadata_df),
            dtype=bool,
        ),
    ),
]

for source_class in [
    "promoter",
    "non-promoter",
]:
    for mutation_level in [4, 8]:
        mask = (
            (
                source_metadata_df[
                    "source_class"
                ] == source_class
            )
            & (
                source_metadata_df[
                    "mutation_level"
                ] == mutation_level
            )
        ).to_numpy()

        analysis_groups.append(
            (
                f"{source_class}_m{mutation_level}",
                mask,
            )
        )


for delta_type, full_matrix in (
    source_delta_matrices.items()
):
    for group_name, group_mask in (
        analysis_groups
    ):
        group_matrix = full_matrix[
            group_mask
        ]

        for center in [False, True]:
            analysis_name = (
                "centered"
                if center
                else "uncentered"
            )

            result = analyze_svd(
                group_matrix,
                center=center,
            )

            result_key = (
                delta_type,
                group_name,
                analysis_name,
            )

            svd_results[result_key] = result

            cumulative_energy = result[
                "cumulative_energy"
            ]

            top_component = result[
                "right_vectors"
            ][0]

            top_component_alignment = float(
                abs(
                    top_component
                    @ promoter_direction_unit
                )
            )

            mean_vector = result[
                "mean_vector"
            ]

            mean_vector_norm = float(
                np.linalg.norm(mean_vector)
            )

            mean_direction_alignment = (
                float(
                    abs(
                        mean_vector
                        @ promoter_direction_unit
                    )
                    / mean_vector_norm
                )
                if mean_vector_norm > 0
                else np.nan
            )

            summary_row = {
                "delta_type": delta_type,
                "group": group_name,
                "analysis": analysis_name,
                "number_of_vectors": int(
                    len(group_matrix)
                ),
                "participation_ratio": float(
                    result[
                        "participation_ratio"
                    ]
                ),
                "rank_for_50_percent": (
                    rank_needed(
                        cumulative_energy,
                        0.50,
                    )
                ),
                "rank_for_80_percent": (
                    rank_needed(
                        cumulative_energy,
                        0.80,
                    )
                ),
                "rank_for_90_percent": (
                    rank_needed(
                        cumulative_energy,
                        0.90,
                    )
                ),
                "rank_for_95_percent": (
                    rank_needed(
                        cumulative_energy,
                        0.95,
                    )
                ),
                "top_component_alignment_with_promoter_direction":
                    top_component_alignment,
                "mean_vector_norm":
                    mean_vector_norm,
                "mean_vector_alignment_with_promoter_direction":
                    mean_direction_alignment,
            }

            for k in K_VALUES:
                if k <= len(cumulative_energy):
                    summary_row[
                        f"energy_top_{k}"
                    ] = float(
                        cumulative_energy[
                            k - 1
                        ]
                    )

            svd_summary_rows.append(
                summary_row
            )

            maximum_components = min(
                32,
                len(
                    result[
                        "singular_values"
                    ]
                ),
            )

            for component_index in range(
                maximum_components
            ):
                component_vector = result[
                    "right_vectors"
                ][component_index]

                component_rows.append({
                    "delta_type":
                        delta_type,
                    "group":
                        group_name,
                    "analysis":
                        analysis_name,
                    "component":
                        component_index + 1,
                    "singular_value":
                        float(
                            result[
                                "singular_values"
                            ][component_index]
                        ),
                    "energy_fraction":
                        float(
                            result[
                                "energy_fraction"
                            ][component_index]
                        ),
                    "cumulative_energy":
                        float(
                            result[
                                "cumulative_energy"
                            ][component_index]
                        ),
                    "absolute_alignment_with_promoter_direction":
                        float(
                            abs(
                                component_vector
                                @ promoter_direction_unit
                            )
                        ),
                })


svd_summary_df = pd.DataFrame(
    svd_summary_rows
)

svd_component_df = pd.DataFrame(
    component_rows
)


# ------------------------------------------------------------
# Display the central GC-up versus GC-down result
# ------------------------------------------------------------

primary_svd_summary = (
    svd_summary_df[
        (
            svd_summary_df[
                "delta_type"
            ] == "gc_increase_vs_decrease"
        )
        & (
            svd_summary_df[
                "group"
            ].isin(
                [
                    "all",
                    "promoter_m4",
                    "promoter_m8",
                    "non-promoter_m4",
                    "non-promoter_m8",
                ]
            )
        )
    ][
        [
            "group",
            "analysis",
            "number_of_vectors",
            "participation_ratio",
            "energy_top_1",
            "energy_top_2",
            "energy_top_4",
            "energy_top_8",
            "energy_top_16",
            "rank_for_80_percent",
            "rank_for_90_percent",
            "rank_for_95_percent",
            "top_component_alignment_with_promoter_direction",
            "mean_vector_alignment_with_promoter_direction",
        ]
    ]
    .sort_values(
        [
            "group",
            "analysis",
        ]
    )
)


print(
    "GC-increase versus GC-decrease "
    "low-rank summary:"
)

display(primary_svd_summary)


print(
    "\nFirst 10 components for the pooled "
    "uncentered GC-up versus GC-down vectors:"
)

display(
    svd_component_df[
        (
            svd_component_df[
                "delta_type"
            ] == "gc_increase_vs_decrease"
        )
        & (
            svd_component_df[
                "group"
            ] == "all"
        )
        & (
            svd_component_df[
                "analysis"
            ] == "uncentered"
        )
    ].head(10)
)


# ------------------------------------------------------------
# Save results and reusable subspace components
# ------------------------------------------------------------

svd_summary_df.to_csv(
    NOTEBOOK7_DIR
    / "source_averaged_svd_summary.csv",
    index=False,
)

svd_component_df.to_csv(
    NOTEBOOK7_DIR
    / "source_averaged_svd_components.csv",
    index=False,
)

source_metadata_df.to_csv(
    NOTEBOOK7_DIR
    / "source_averaged_delta_metadata.csv",
    index=False,
)

np.savez_compressed(
    NOTEBOOK7_DIR
    / "source_averaged_delta_matrices.npz",
    gc_increase_vs_neutral=(
        source_gc_up_vs_neutral
    ),
    gc_decrease_vs_neutral=(
        source_gc_down_vs_neutral
    ),
    gc_increase_vs_decrease=(
        source_gc_up_vs_down
    ),
)

primary_result = svd_results[
    (
        "gc_increase_vs_decrease",
        "all",
        "uncentered",
    )
]

np.savez_compressed(
    NOTEBOOK7_DIR
    / "gc_directional_subspace_uncentered.npz",
    mean_vector=(
        primary_result[
            "mean_vector"
        ]
    ),
    singular_values=(
        primary_result[
            "singular_values"
        ]
    ),
    right_vectors=(
        primary_result[
            "right_vectors"
        ]
    ),
    energy_fraction=(
        primary_result[
            "energy_fraction"
        ]
    ),
    cumulative_energy=(
        primary_result[
            "cumulative_energy"
        ]
    ),
)


print(
    "\nSaved source-averaged SVD results to:"
)

print(NOTEBOOK7_DIR)

print(
    "\nNotebook 7 Cell 2 completed."
)

Source-averaged metadata rows: 756
Source-averaged matrix shape: (756, 768)
GC-increase versus GC-decrease low-rank summary:


,group,analysis,number_of_vectors,participation_ratio,energy_top_1,energy_top_2,energy_top_4,energy_top_8,energy_top_16,rank_for_80_percent,rank_for_90_percent,rank_for_95_percent,top_component_alignment_with_promoter_direction,mean_vector_alignment_with_promoter_direction
21,all,centered,756,7.274415,0.357456,0.417764,0.492090,0.563907,0.643954,52,111,179,0.756273,0.893683
20,all,uncentered,756,2.988808,0.570767,0.651323,0.698652,0.743450,0.790609,19,64,126,0.894321,0.893683
27,non-promoter_m4,centered,189,26.486905,0.152335,0.217624,0.300812,0.392928,0.512140,57,91,120,0.760465,0.839178
26,non-promoter_m4,uncentered,189,6.632058,0.375762,0.431244,0.508946,0.578215,0.662793,39,73,105,0.871977,0.839178
29,non-promoter_m8,centered,189,13.594346,0.246985,0.305815,0.388274,0.478505,0.590499,48,80,111,0.791234,0.838681
28,non-promoter_m8,uncentered,189,3.107520,0.560327,0.630295,0.680986,0.732134,0.791688,18,48,81,0.871285,0.838681
23,promoter_m4,centered,189,11.532845,0.273246,0.330033,0.415098,0.501424,0.606311,45,78,108,0.624414,0.897854
22,promoter_m4,uncentered,189,3.516657,0.526279,0.588845,0.641725,0.699898,0.764354,23,53,86,0.871334,0.897854
25,promoter_m8,centered,189,4.927115,0.439709,0.506318,0.570409,0.639535,0.722183,29,60,92,0.645193,0.896707
24,promoter_m8,uncentered,189,2.064772,0.691533,0.759946,0.796367,0.832693,0.872172,5,26,56,0.873099,0.896707



First 10 components for the pooled uncentered GC-up versus GC-down vectors:


,delta_type,group,analysis,component,singular_value,energy_fraction,cumulative_energy,absolute_alignment_with_promoter_direction
640,gc_increase_vs_decrease,all,uncentered,1,19.119622,0.570767,0.570767,0.894321
641,gc_increase_vs_decrease,all,uncentered,2,7.182851,0.080555,0.651323,0.103079
642,gc_increase_vs_decrease,all,uncentered,3,4.229819,0.027935,0.679257,0.239961
643,gc_increase_vs_decrease,all,uncentered,4,3.524413,0.019394,0.698652,0.052080
644,gc_increase_vs_decrease,all,uncentered,5,3.154417,0.015536,0.714188,0.058627
645,gc_increase_vs_decrease,all,uncentered,6,2.743847,0.011755,0.725943,0.001741
646,gc_increase_vs_decrease,all,uncentered,7,2.468961,0.009518,0.735460,0.064412
647,gc_increase_vs_decrease,all,uncentered,8,2.262168,0.007990,0.743450,0.056864
648,gc_increase_vs_decrease,all,uncentered,9,2.194420,0.007519,0.750969,0.001962
649,gc_increase_vs_decrease,all,uncentered,10,2.072043,0.006703,0.757672,0.051202



Saved source-averaged SVD results to:
/content/drive/MyDrive/algoverse/dnabert/outputs/promoter_mlp_low_rank_subspace

Notebook 7 Cell 2 completed.


Cell 3: cross-validated held-out subspace reconstruction

In [5]:
# Test whether the GC-responsive low-rank subspace generalizes
# to held-out source sequences

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

N_SPLITS = 5

K_VALUES = [
    1,
    2,
    4,
    8,
    16,
    32,
    64,
    128,
]


# Use the direct GC-up versus GC-down contrast as the
# primary directional intervention.
X = source_gc_up_vs_down.astype(
    np.float64
)

metadata = source_metadata_df.copy()


# Keep the same biological source in one fold across
# its 4-mutation and 8-mutation conditions.
metadata[
    "source_group"
] = (
    metadata["source_class"].astype(str)
    + "_"
    + metadata["source_index"].astype(str)
)

groups = metadata[
    "source_group"
].to_numpy()


print("Input matrix shape:", X.shape)

print(
    "Unique biological source groups:",
    metadata["source_group"].nunique(),
)

assert X.shape == (756, 768)

assert (
    metadata[
        "source_group"
    ].nunique()
    == 378
)


# ------------------------------------------------------------
# Reconstruction helpers
# ------------------------------------------------------------

def project_and_reconstruct(
    matrix,
    components,
):
    """
    Orthogonally project vectors into a subspace and
    reconstruct them in the original 768-dimensional space.
    """

    coordinates = (
        matrix
        @ components.T
    )

    reconstruction = (
        coordinates
        @ components
    )

    return coordinates, reconstruction


def row_energy_fraction(
    original,
    reconstruction,
):
    """
    Fraction of each vector's squared norm captured by
    its projection into the learned subspace.
    """

    original_energy = np.sum(
        original ** 2,
        axis=1,
    )

    reconstruction_energy = np.sum(
        reconstruction ** 2,
        axis=1,
    )

    return np.divide(
        reconstruction_energy,
        original_energy,
        out=np.full_like(
            original_energy,
            np.nan,
            dtype=float,
        ),
        where=original_energy > 0,
    )


def row_cosine_similarity(
    original,
    reconstruction,
):
    numerator = np.sum(
        original * reconstruction,
        axis=1,
    )

    denominator = (
        np.linalg.norm(
            original,
            axis=1,
        )
        * np.linalg.norm(
            reconstruction,
            axis=1,
        )
    )

    return np.divide(
        numerator,
        denominator,
        out=np.full_like(
            numerator,
            np.nan,
            dtype=float,
        ),
        where=denominator > 0,
    )


# ------------------------------------------------------------
# Grouped cross-validation
# ------------------------------------------------------------

group_cv = GroupKFold(
    n_splits=N_SPLITS
)

heldout_rows = []
fold_component_rows = []


for fold_index, (
    train_indices,
    test_indices,
) in enumerate(
    group_cv.split(
        X,
        groups=groups,
    ),
    start=1,
):
    X_train = X[
        train_indices
    ]

    X_test = X[
        test_indices
    ]

    # Uncentered SVD preserves the shared mean GC effect.
    _, train_singular_values, train_components = (
        np.linalg.svd(
            X_train,
            full_matrices=False,
        )
    )

    # Save fold-level component properties.
    for component_index in range(
        min(16, len(train_components))
    ):
        component = train_components[
            component_index
        ]

        fold_component_rows.append({
            "fold": fold_index,
            "component": component_index + 1,
            "singular_value": float(
                train_singular_values[
                    component_index
                ]
            ),
            "absolute_alignment_with_promoter_direction":
                float(
                    abs(
                        component
                        @ promoter_direction_unit
                    )
                ),
        })


    for k in K_VALUES:
        if k > train_components.shape[0]:
            continue

        components_k = train_components[
            :k
        ]

        (
            test_coordinates,
            test_reconstruction,
        ) = project_and_reconstruct(
            X_test,
            components_k,
        )

        energy_fractions = row_energy_fraction(
            X_test,
            test_reconstruction,
        )

        cosine_similarities = (
            row_cosine_similarity(
                X_test,
                test_reconstruction,
            )
        )

        original_promoter_projection = (
            X_test
            @ promoter_direction_unit
        )

        reconstructed_promoter_projection = (
            test_reconstruction
            @ promoter_direction_unit
        )

        for local_index, global_index in enumerate(
            test_indices
        ):
            heldout_rows.append({
                "fold": fold_index,
                "k": k,
                "source_class": metadata.iloc[
                    global_index
                ]["source_class"],
                "source_index": int(
                    metadata.iloc[
                        global_index
                    ]["source_index"]
                ),
                "mutation_level": int(
                    metadata.iloc[
                        global_index
                    ]["mutation_level"]
                ),
                "source_group": metadata.iloc[
                    global_index
                ]["source_group"],
                "captured_energy_fraction": float(
                    energy_fractions[
                        local_index
                    ]
                ),
                "reconstruction_cosine": float(
                    cosine_similarities[
                        local_index
                    ]
                ),
                "original_promoter_projection": float(
                    original_promoter_projection[
                        local_index
                    ]
                ),
                "reconstructed_promoter_projection": float(
                    reconstructed_promoter_projection[
                        local_index
                    ]
                ),
                "promoter_projection_fraction": (
                    float(
                        reconstructed_promoter_projection[
                            local_index
                        ]
                        / original_promoter_projection[
                            local_index
                        ]
                    )
                    if abs(
                        original_promoter_projection[
                            local_index
                        ]
                    ) > 1e-12
                    else np.nan
                ),
            })


heldout_reconstruction_df = pd.DataFrame(
    heldout_rows
)

fold_component_df = pd.DataFrame(
    fold_component_rows
)


# ------------------------------------------------------------
# Aggregate held-out performance
# ------------------------------------------------------------

heldout_summary_df = (
    heldout_reconstruction_df
    .groupby(
        [
            "source_class",
            "mutation_level",
            "k",
        ]
    )
    .agg(
        number_of_vectors=(
            "captured_energy_fraction",
            "size",
        ),
        mean_captured_energy=(
            "captured_energy_fraction",
            "mean",
        ),
        median_captured_energy=(
            "captured_energy_fraction",
            "median",
        ),
        mean_reconstruction_cosine=(
            "reconstruction_cosine",
            "mean",
        ),
        median_reconstruction_cosine=(
            "reconstruction_cosine",
            "median",
        ),
        mean_original_promoter_projection=(
            "original_promoter_projection",
            "mean",
        ),
        mean_reconstructed_promoter_projection=(
            "reconstructed_promoter_projection",
            "mean",
        ),
    )
    .reset_index()
)


pooled_heldout_summary_df = (
    heldout_reconstruction_df
    .groupby("k")
    .agg(
        number_of_vectors=(
            "captured_energy_fraction",
            "size",
        ),
        mean_captured_energy=(
            "captured_energy_fraction",
            "mean",
        ),
        median_captured_energy=(
            "captured_energy_fraction",
            "median",
        ),
        mean_reconstruction_cosine=(
            "reconstruction_cosine",
            "mean",
        ),
        median_reconstruction_cosine=(
            "reconstruction_cosine",
            "median",
        ),
        mean_original_promoter_projection=(
            "original_promoter_projection",
            "mean",
        ),
        mean_reconstructed_promoter_projection=(
            "reconstructed_promoter_projection",
            "mean",
        ),
    )
    .reset_index()
)


fold_pc1_summary_df = (
    fold_component_df[
        fold_component_df[
            "component"
        ] == 1
    ]
    .copy()
)


print(
    "Held-out pooled subspace reconstruction:"
)

display(
    pooled_heldout_summary_df
)


print(
    "\nHeld-out reconstruction by class and dose:"
)

display(
    heldout_summary_df[
        heldout_summary_df[
            "k"
        ].isin(
            [1, 4, 8, 16, 32]
        )
    ]
)


print(
    "\nTraining-fold PC1 alignment with "
    "the promoter direction:"
)

display(fold_pc1_summary_df)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

# [staged fix] this key omits mutation_level; the next cell validates on the correct key and saves.
try:
    assert (
        heldout_reconstruction_df[
            ["source_group", "k"]
        ]
        .duplicated()
        .sum()
        == 0
    )

    assert (
        heldout_reconstruction_df[
            "captured_energy_fraction"
        ]
        .between(
            -1e-8,
            1 + 1e-8,
        )
        .all()
    )

    assert (
        heldout_reconstruction_df[
            "reconstruction_cosine"
        ]
        .between(
            -1 - 1e-8,
            1 + 1e-8,
        )
        .all()
    )


    # ------------------------------------------------------------
    # Save results
    # ------------------------------------------------------------

    heldout_reconstruction_df.to_csv(
        NOTEBOOK7_DIR
        / "heldout_subspace_reconstruction_rows.csv",
        index=False,
    )

    heldout_summary_df.to_csv(
        NOTEBOOK7_DIR
        / "heldout_subspace_reconstruction_by_group.csv",
        index=False,
    )

    pooled_heldout_summary_df.to_csv(
        NOTEBOOK7_DIR
        / "heldout_subspace_reconstruction_pooled.csv",
        index=False,
    )

    fold_component_df.to_csv(
        NOTEBOOK7_DIR
        / "cross_validated_subspace_components.csv",
        index=False,
    )


    print(
        "\nSaved held-out subspace results to:"
    )

    print(NOTEBOOK7_DIR)

    print(
        "\nNotebook 7 Cell 3 completed."
    )
except Exception as exc:
    print("[staged fix] superseded step did not run:", type(exc).__name__, exc)
    print("[staged fix] the following cell redoes this correctly.")


Input matrix shape: (756, 768)
Unique biological source groups: 378
Held-out pooled subspace reconstruction:


,k,number_of_vectors,mean_captured_energy,median_captured_energy,mean_reconstruction_cosine,median_reconstruction_cosine,mean_original_promoter_projection,mean_reconstructed_promoter_projection
0,1,756,0.446160,0.487607,0.624079,0.698288,0.528381,0.519052
1,2,756,0.537878,0.561359,0.715106,0.749237,0.528381,0.529969
2,4,756,0.590516,0.606159,0.757227,0.778562,0.528381,0.527268
3,8,756,0.640865,0.659139,0.792941,0.811874,0.528381,0.527720
4,16,756,0.690300,0.708229,0.825620,0.841564,0.528381,0.527939
5,32,756,0.748888,0.762380,0.862182,0.873143,0.528381,0.528292
6,64,756,0.811373,0.825940,0.899057,0.908813,0.528381,0.528241
7,128,756,0.876159,0.885010,0.935348,0.940750,0.528381,0.528245



Held-out reconstruction by class and dose:


,source_class,mutation_level,k,number_of_vectors,mean_captured_energy,median_captured_energy,mean_reconstruction_cosine,median_reconstruction_cosine,mean_original_promoter_projection,mean_reconstructed_promoter_projection
0,non-promoter,4,1,189,0.302335,0.275410,0.508271,0.524795,0.284755,0.272603
2,non-promoter,4,4,189,0.450636,0.451898,0.661377,0.672234,0.284755,0.283430
3,non-promoter,4,8,189,0.516126,0.521717,0.712181,0.722300,0.284755,0.283650
4,non-promoter,4,16,189,0.575943,0.581197,0.754901,0.762362,0.284755,0.284163
5,non-promoter,4,32,189,0.650385,0.660717,0.804019,0.812845,0.284755,0.284281
8,non-promoter,8,1,189,0.455201,0.495825,0.641950,0.704148,0.577825,0.562722
10,non-promoter,8,4,189,0.634031,0.640234,0.791545,0.800146,0.577825,0.575099
11,non-promoter,8,8,189,0.681055,0.698689,0.822206,0.835876,0.577825,0.575950
12,non-promoter,8,16,189,0.727672,0.740989,0.851139,0.860807,0.577825,0.576110
13,non-promoter,8,32,189,0.780625,0.786658,0.882436,0.886937,0.577825,0.576210



Training-fold PC1 alignment with the promoter direction:


,fold,component,singular_value,absolute_alignment_with_promoter_direction
0,1,1,17.047690,0.895019
16,2,1,17.073059,0.894815
32,3,1,16.978061,0.897109
48,4,1,17.180437,0.892707
64,5,1,17.240074,0.891207


[staged fix] superseded step did not run: AssertionError 
[staged fix] the following cell redoes this correctly.


In [6]:

# Correct validation and save held-out subspace results

# Each biological source appears once at mutation level 4
# and once at mutation level 8 for every value of k.
# Therefore mutation_level must be part of the unique key.

duplicate_count = (
    heldout_reconstruction_df[
        [
            "source_group",
            "mutation_level",
            "k",
        ]
    ]
    .duplicated()
    .sum()
)

print(
    "Duplicate held-out rows under the correct key:",
    duplicate_count,
)

assert duplicate_count == 0


# Confirm expected row counts:
# 756 source/dose vectors × 8 subspace sizes.
expected_row_count = (
    len(source_metadata_df)
    * len(K_VALUES)
)

print(
    "Held-out reconstruction rows:",
    len(heldout_reconstruction_df),
)

print(
    "Expected reconstruction rows:",
    expected_row_count,
)

assert (
    len(heldout_reconstruction_df)
    == expected_row_count
)


# Every source/dose vector should appear once for each k.
rows_per_source_dose = (
    heldout_reconstruction_df
    .groupby(
        [
            "source_group",
            "mutation_level",
        ]
    )
    .size()
)

assert rows_per_source_dose.eq(
    len(K_VALUES)
).all()


# Numerical range checks.
assert (
    heldout_reconstruction_df[
        "captured_energy_fraction"
    ]
    .between(
        -1e-8,
        1 + 1e-8,
    )
    .all()
)

assert (
    heldout_reconstruction_df[
        "reconstruction_cosine"
    ]
    .between(
        -1 - 1e-8,
        1 + 1e-8,
    )
    .all()
)


# Confirm that all five folds contain disjoint source groups.
fold_source_sets = {
    fold: set(
        group_df[
            "source_group"
        ].unique()
    )
    for fold, group_df in (
        heldout_reconstruction_df[
            heldout_reconstruction_df[
                "k"
            ] == K_VALUES[0]
        ]
        .groupby("fold")
    )
}

for first_fold in fold_source_sets:
    for second_fold in fold_source_sets:
        if first_fold >= second_fold:
            continue

        assert fold_source_sets[
            first_fold
        ].isdisjoint(
            fold_source_sets[
                second_fold
            ]
        )


# Compact diagnostic: fraction of the mean promoter-direction
# shift retained by each held-out subspace.
pooled_heldout_summary_df[
    "mean_promoter_projection_retained"
] = (
    pooled_heldout_summary_df[
        "mean_reconstructed_promoter_projection"
    ]
    / pooled_heldout_summary_df[
        "mean_original_promoter_projection"
    ]
)


print(
    "\nCorrected held-out pooled summary:"
)

display(
    pooled_heldout_summary_df[
        [
            "k",
            "number_of_vectors",
            "mean_captured_energy",
            "median_captured_energy",
            "mean_reconstruction_cosine",
            "mean_original_promoter_projection",
            "mean_reconstructed_promoter_projection",
            "mean_promoter_projection_retained",
        ]
    ]
)


# Save all results that were skipped after the original assertion.
heldout_reconstruction_df.to_csv(
    NOTEBOOK7_DIR
    / "heldout_subspace_reconstruction_rows.csv",
    index=False,
)

heldout_summary_df.to_csv(
    NOTEBOOK7_DIR
    / "heldout_subspace_reconstruction_by_group.csv",
    index=False,
)

pooled_heldout_summary_df.to_csv(
    NOTEBOOK7_DIR
    / "heldout_subspace_reconstruction_pooled.csv",
    index=False,
)

fold_component_df.to_csv(
    NOTEBOOK7_DIR
    / "cross_validated_subspace_components.csv",
    index=False,
)


print(
    "\nSaved corrected held-out subspace results to:"
)

for filename in [
    "heldout_subspace_reconstruction_rows.csv",
    "heldout_subspace_reconstruction_by_group.csv",
    "heldout_subspace_reconstruction_pooled.csv",
    "cross_validated_subspace_components.csv",
]:
    print(
        NOTEBOOK7_DIR / filename
    )

print(
    "\nNotebook 7 Cell 3 completed successfully."
)

Duplicate held-out rows under the correct key: 0
Held-out reconstruction rows: 6048
Expected reconstruction rows: 6048

Corrected held-out pooled summary:


,k,number_of_vectors,mean_captured_energy,median_captured_energy,mean_reconstruction_cosine,mean_original_promoter_projection,mean_reconstructed_promoter_projection,mean_promoter_projection_retained
0,1,756,0.446160,0.487607,0.624079,0.528381,0.519052,0.982344
1,2,756,0.537878,0.561359,0.715106,0.528381,0.529969,1.003004
2,4,756,0.590516,0.606159,0.757227,0.528381,0.527268,0.997894
3,8,756,0.640865,0.659139,0.792941,0.528381,0.527720,0.998749
4,16,756,0.690300,0.708229,0.825620,0.528381,0.527939,0.999163
5,32,756,0.748888,0.762380,0.862182,0.528381,0.528292,0.999830
6,64,756,0.811373,0.825940,0.899057,0.528381,0.528241,0.999733
7,128,756,0.876159,0.885010,0.935348,0.528381,0.528245,0.999742



Saved corrected held-out subspace results to:
/content/drive/MyDrive/algoverse/dnabert/outputs/promoter_mlp_low_rank_subspace/heldout_subspace_reconstruction_rows.csv
/content/drive/MyDrive/algoverse/dnabert/outputs/promoter_mlp_low_rank_subspace/heldout_subspace_reconstruction_by_group.csv
/content/drive/MyDrive/algoverse/dnabert/outputs/promoter_mlp_low_rank_subspace/heldout_subspace_reconstruction_pooled.csv
/content/drive/MyDrive/algoverse/dnabert/outputs/promoter_mlp_low_rank_subspace/cross_validated_subspace_components.csv

Notebook 7 Cell 3 completed successfully.


Cell 4: predict downstream probe effects from held-out low-rank coordinates

In [7]:
# Test whether low-rank Layer-1 MLP intervention coordinates
# predict the downstream Layer-5 probe-logit contrast

import numpy as np
import pandas as pd

from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from scipy.stats import pearsonr, spearmanr


# ------------------------------------------------------------
# Construct the downstream GC-up versus GC-down contrast
# ------------------------------------------------------------

downstream_block_df = (
    gc_intervention_scored_df[
        gc_intervention_scored_df[
            "intervention_type"
        ].isin(
            [
                "gc_increased",
                "gc_decreased",
            ]
        )
    ]
    .pivot(
        index=[
            "source_class",
            "source_index",
            "mutation_level",
            "variant_seed",
        ],
        columns="intervention_type",
        values="probe_logit_change",
    )
    .reset_index()
)

downstream_block_df.columns.name = None

downstream_block_df[
    "probe_gc_increase_vs_decrease"
] = (
    downstream_block_df[
        "gc_increased"
    ]
    - downstream_block_df[
        "gc_decreased"
    ]
)


# Average the five mutation seeds for each source and dose.
downstream_source_df = (
    downstream_block_df
    .groupby(
        [
            "source_class",
            "source_index",
            "mutation_level",
        ],
        as_index=False,
    )
    .agg(
        probe_gc_increase_vs_decrease=(
            "probe_gc_increase_vs_decrease",
            "mean",
        )
    )
)


# Align downstream effects exactly with the source-level
# Layer-1 MLP matrix created in Cell 2.
modeling_df = (
    source_metadata_df
    .merge(
        downstream_source_df,
        on=[
            "source_class",
            "source_index",
            "mutation_level",
        ],
        how="left",
        validate="one_to_one",
    )
)

assert len(modeling_df) == len(
    source_gc_up_vs_down
)

assert modeling_df[
    "probe_gc_increase_vs_decrease"
].notna().all()


X = source_gc_up_vs_down.astype(
    np.float64
)

y = modeling_df[
    "probe_gc_increase_vs_decrease"
].to_numpy(dtype=np.float64)

groups = (
    modeling_df["source_class"].astype(str)
    + "_"
    + modeling_df["source_index"].astype(str)
).to_numpy()


print("Layer-1 intervention matrix:", X.shape)
print("Downstream target shape:", y.shape)
print("Unique biological sources:", len(np.unique(groups)))

print(
    "\nMean downstream GC-up versus GC-down effect:",
    y.mean(),
)

print(
    "Standard deviation:",
    y.std(ddof=1),
)


# ------------------------------------------------------------
# Cross-validated low-rank prediction
# ------------------------------------------------------------

K_VALUES_MODEL = [
    1,
    2,
    4,
    8,
    16,
    32,
    64,
]

RIDGE_ALPHA = 1.0

group_cv = GroupKFold(
    n_splits=5
)

prediction_rows = []


for fold_index, (
    train_indices,
    test_indices,
) in enumerate(
    group_cv.split(
        X,
        y,
        groups=groups,
    ),
    start=1,
):
    X_train = X[train_indices]
    X_test = X[test_indices]

    y_train = y[train_indices]
    y_test = y[test_indices]

    # Learn the intervention subspace only from training sources.
    _, _, train_components = np.linalg.svd(
        X_train,
        full_matrices=False,
    )

    # Baseline: predict the training mean.
    baseline_prediction = np.full(
        len(test_indices),
        y_train.mean(),
    )

    for local_index, global_index in enumerate(
        test_indices
    ):
        prediction_rows.append({
            "fold": fold_index,
            "model": "training_mean",
            "k": 0,
            "source_class": modeling_df.iloc[
                global_index
            ]["source_class"],
            "source_index": int(
                modeling_df.iloc[
                    global_index
                ]["source_index"]
            ),
            "mutation_level": int(
                modeling_df.iloc[
                    global_index
                ]["mutation_level"]
            ),
            "observed_probe_effect": float(
                y_test[local_index]
            ),
            "predicted_probe_effect": float(
                baseline_prediction[
                    local_index
                ]
            ),
        })


    # Promoter-direction scalar baseline.
    promoter_train_feature = (
        X_train
        @ promoter_direction_unit
    ).reshape(-1, 1)

    promoter_test_feature = (
        X_test
        @ promoter_direction_unit
    ).reshape(-1, 1)

    promoter_scaler = StandardScaler()

    promoter_train_scaled = (
        promoter_scaler.fit_transform(
            promoter_train_feature
        )
    )

    promoter_test_scaled = (
        promoter_scaler.transform(
            promoter_test_feature
        )
    )

    promoter_model = Ridge(
        alpha=RIDGE_ALPHA
    )

    promoter_model.fit(
        promoter_train_scaled,
        y_train,
    )

    promoter_predictions = (
        promoter_model.predict(
            promoter_test_scaled
        )
    )

    for local_index, global_index in enumerate(
        test_indices
    ):
        prediction_rows.append({
            "fold": fold_index,
            "model": "promoter_direction",
            "k": 1,
            "source_class": modeling_df.iloc[
                global_index
            ]["source_class"],
            "source_index": int(
                modeling_df.iloc[
                    global_index
                ]["source_index"]
            ),
            "mutation_level": int(
                modeling_df.iloc[
                    global_index
                ]["mutation_level"]
            ),
            "observed_probe_effect": float(
                y_test[local_index]
            ),
            "predicted_probe_effect": float(
                promoter_predictions[
                    local_index
                ]
            ),
        })


    # SVD coordinate models.
    for k in K_VALUES_MODEL:
        components_k = train_components[
            :k
        ]

        train_coordinates = (
            X_train
            @ components_k.T
        )

        test_coordinates = (
            X_test
            @ components_k.T
        )

        coordinate_scaler = StandardScaler()

        train_coordinates_scaled = (
            coordinate_scaler.fit_transform(
                train_coordinates
            )
        )

        test_coordinates_scaled = (
            coordinate_scaler.transform(
                test_coordinates
            )
        )

        ridge_model = Ridge(
            alpha=RIDGE_ALPHA
        )

        ridge_model.fit(
            train_coordinates_scaled,
            y_train,
        )

        test_predictions = ridge_model.predict(
            test_coordinates_scaled
        )

        for local_index, global_index in enumerate(
            test_indices
        ):
            prediction_rows.append({
                "fold": fold_index,
                "model": "svd_coordinates",
                "k": k,
                "source_class": modeling_df.iloc[
                    global_index
                ]["source_class"],
                "source_index": int(
                    modeling_df.iloc[
                        global_index
                    ]["source_index"]
                ),
                "mutation_level": int(
                    modeling_df.iloc[
                        global_index
                    ]["mutation_level"]
                ),
                "observed_probe_effect": float(
                    y_test[local_index]
                ),
                "predicted_probe_effect": float(
                    test_predictions[
                        local_index
                    ]
                ),
            })


prediction_df = pd.DataFrame(
    prediction_rows
)


# ------------------------------------------------------------
# Aggregate held-out prediction performance
# ------------------------------------------------------------

performance_rows = []

for (
    model_name,
    k,
), group_df in prediction_df.groupby(
    [
        "model",
        "k",
    ]
):
    observed = group_df[
        "observed_probe_effect"
    ].to_numpy()

    predicted = group_df[
        "predicted_probe_effect"
    ].to_numpy()

    residual = observed - predicted

    pearson_result = pearsonr(
        observed,
        predicted,
    )

    spearman_result = spearmanr(
        observed,
        predicted,
    )

    performance_rows.append({
        "model": model_name,
        "k": int(k),
        "number_of_vectors": len(
            group_df
        ),
        "out_of_fold_r2": float(
            r2_score(
                observed,
                predicted,
            )
        ),
        "pearson_r": float(
            pearson_result.statistic
        ),
        "spearman_rho": float(
            spearman_result.statistic
        ),
        "mean_absolute_error": float(
            np.mean(
                np.abs(residual)
            )
        ),
        "root_mean_squared_error": float(
            np.sqrt(
                np.mean(
                    residual ** 2
                )
            )
        ),
    })


prediction_performance_df = (
    pd.DataFrame(
        performance_rows
    )
    .sort_values(
        [
            "model",
            "k",
        ]
    )
    .reset_index(drop=True)
)


print(
    "\nHeld-out prediction of downstream "
    "Layer-5 probe effects:"
)

display(prediction_performance_df)


# Performance by source class and dose.
group_performance_rows = []

for (
    model_name,
    k,
    source_class,
    mutation_level,
), group_df in prediction_df.groupby(
    [
        "model",
        "k",
        "source_class",
        "mutation_level",
    ]
):
    observed = group_df[
        "observed_probe_effect"
    ].to_numpy()

    predicted = group_df[
        "predicted_probe_effect"
    ].to_numpy()

    group_performance_rows.append({
        "model": model_name,
        "k": int(k),
        "source_class": source_class,
        "mutation_level": int(
            mutation_level
        ),
        "number_of_vectors": len(
            group_df
        ),
        "r2": float(
            r2_score(
                observed,
                predicted,
            )
        ),
        "pearson_r": float(
            pearsonr(
                observed,
                predicted,
            ).statistic
        ),
        "spearman_rho": float(
            spearmanr(
                observed,
                predicted,
            ).statistic
        ),
    })


prediction_group_performance_df = (
    pd.DataFrame(
        group_performance_rows
    )
)


print(
    "\nPrediction performance by class and dose "
    "for selected models:"
)

display(
    prediction_group_performance_df[
        (
            (
                prediction_group_performance_df[
                    "model"
                ] == "promoter_direction"
            )
        )
        |
        (
            (
                prediction_group_performance_df[
                    "model"
                ] == "svd_coordinates"
            )
            & (
                prediction_group_performance_df[
                    "k"
                ].isin(
                    [1, 4, 8, 16]
                )
            )
        )
    ]
)


# ------------------------------------------------------------
# Validation and saving
# ------------------------------------------------------------

expected_models_per_vector = (
    1
    + 1
    + len(K_VALUES_MODEL)
)

assert len(prediction_df) == (
    len(X)
    * expected_models_per_vector
)

assert prediction_df[
    [
        "model",
        "k",
        "source_class",
        "source_index",
        "mutation_level",
    ]
].duplicated().sum() == 0

assert np.isfinite(
    prediction_df[
        "predicted_probe_effect"
    ]
).all()


prediction_df.to_csv(
    NOTEBOOK7_DIR
    / "heldout_subspace_downstream_predictions.csv",
    index=False,
)

prediction_performance_df.to_csv(
    NOTEBOOK7_DIR
    / "heldout_subspace_downstream_performance.csv",
    index=False,
)

prediction_group_performance_df.to_csv(
    NOTEBOOK7_DIR
    / "heldout_subspace_downstream_performance_by_group.csv",
    index=False,
)


print(
    "\nSaved downstream prediction results to:"
)

print(NOTEBOOK7_DIR)

print(
    "\nNotebook 7 Cell 4 completed."
)

Layer-1 intervention matrix: (756, 768)
Downstream target shape: (756,)
Unique biological sources: 378

Mean downstream GC-up versus GC-down effect: 4.137789567643621
Standard deviation: 4.944641336771111

Held-out prediction of downstream Layer-5 probe effects:


,model,k,number_of_vectors,out_of_fold_r2,pearson_r,spearman_rho,mean_absolute_error,root_mean_squared_error
0,promoter_direction,1,756,0.549133,0.741040,0.720624,2.554062,3.317964
1,svd_coordinates,1,756,0.554241,0.744475,0.706500,2.563958,3.299117
2,svd_coordinates,2,756,0.597488,0.772975,0.735308,2.432622,3.134995
3,svd_coordinates,4,756,0.602831,0.776436,0.749488,2.411743,3.114118
4,svd_coordinates,8,756,0.611631,0.782096,0.759497,2.398865,3.079424
5,svd_coordinates,16,756,0.615998,0.784956,0.760119,2.399218,3.062064
6,svd_coordinates,32,756,0.617712,0.786459,0.764062,2.384016,3.055220
7,svd_coordinates,64,756,0.623145,0.790919,0.754726,2.374224,3.033435
8,training_mean,0,756,-0.002640,-0.068303,-0.060764,3.793472,4.947889



Prediction performance by class and dose for selected models:


,model,k,source_class,mutation_level,number_of_vectors,r2,pearson_r,spearman_rho
0,promoter_direction,1,non-promoter,4,189,0.169409,0.468294,0.517752
1,promoter_direction,1,non-promoter,8,189,0.281012,0.621632,0.659351
2,promoter_direction,1,promoter,4,189,0.455105,0.713111,0.702105
3,promoter_direction,1,promoter,8,189,0.591117,0.779058,0.816833
4,svd_coordinates,1,non-promoter,4,189,0.117437,0.414304,0.453807
5,svd_coordinates,1,non-promoter,8,189,0.268491,0.598039,0.619997
6,svd_coordinates,1,promoter,4,189,0.449442,0.695230,0.671971
7,svd_coordinates,1,promoter,8,189,0.621915,0.796635,0.820864
12,svd_coordinates,4,non-promoter,4,189,0.243569,0.497499,0.544750
13,svd_coordinates,4,non-promoter,8,189,0.396901,0.641064,0.659159



Saved downstream prediction results to:
/content/drive/MyDrive/algoverse/dnabert/outputs/promoter_mlp_low_rank_subspace

Notebook 7 Cell 4 completed.


# Cell 5
# Finalize low-rank subspace results and save summary

In [8]:
import json

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Extract key results
# ------------------------------------------------------------

def get_prediction_result(
    model,
    k,
):
    row = prediction_performance_df[
        (
            prediction_performance_df[
                "model"
            ] == model
        )
        & (
            prediction_performance_df[
                "k"
            ] == k
        )
    ]

    if len(row) != 1:
        raise ValueError(
            f"Expected one result for {model}, k={k}."
        )

    return row.iloc[0]


def get_heldout_reconstruction_result(
    k,
):
    row = pooled_heldout_summary_df[
        pooled_heldout_summary_df[
            "k"
        ] == k
    ]

    if len(row) != 1:
        raise ValueError(
            f"Expected one reconstruction result for k={k}."
        )

    return row.iloc[0]


promoter_direction_result = (
    get_prediction_result(
        "promoter_direction",
        1,
    )
)

svd_1_result = get_prediction_result(
    "svd_coordinates",
    1,
)

svd_4_result = get_prediction_result(
    "svd_coordinates",
    4,
)

svd_8_result = get_prediction_result(
    "svd_coordinates",
    8,
)

svd_16_result = get_prediction_result(
    "svd_coordinates",
    16,
)

svd_64_result = get_prediction_result(
    "svd_coordinates",
    64,
)


reconstruction_1 = (
    get_heldout_reconstruction_result(1)
)

reconstruction_4 = (
    get_heldout_reconstruction_result(4)
)

reconstruction_8 = (
    get_heldout_reconstruction_result(8)
)

reconstruction_16 = (
    get_heldout_reconstruction_result(16)
)


# Pooled in-sample SVD result from Cell 2.
primary_svd_row = (
    svd_summary_df[
        (
            svd_summary_df[
                "delta_type"
            ] == "gc_increase_vs_decrease"
        )
        & (
            svd_summary_df[
                "group"
            ] == "all"
        )
        & (
            svd_summary_df[
                "analysis"
            ] == "uncentered"
        )
    ]
)

assert len(primary_svd_row) == 1

primary_svd_row = primary_svd_row.iloc[0]


# ------------------------------------------------------------
# Compact final tables
# ------------------------------------------------------------

final_subspace_summary_df = pd.DataFrame([
    {
        "analysis": "pooled_uncentered_svd",
        "metric": "PC1 energy fraction",
        "value": float(
            primary_svd_row["energy_top_1"]
        ),
    },
    {
        "analysis": "pooled_uncentered_svd",
        "metric": "participation ratio",
        "value": float(
            primary_svd_row[
                "participation_ratio"
            ]
        ),
    },
    {
        "analysis": "pooled_uncentered_svd",
        "metric": (
            "PC1 alignment with promoter direction"
        ),
        "value": float(
            primary_svd_row[
                "top_component_alignment_with_promoter_direction"
            ]
        ),
    },
    {
        "analysis": "heldout_reconstruction",
        "metric": "PC1 captured energy",
        "value": float(
            reconstruction_1[
                "mean_captured_energy"
            ]
        ),
    },
    {
        "analysis": "heldout_reconstruction",
        "metric": (
            "PC1 promoter projection retained"
        ),
        "value": float(
            reconstruction_1[
                "mean_promoter_projection_retained"
            ]
        ),
    },
    {
        "analysis": "heldout_reconstruction",
        "metric": "4D captured energy",
        "value": float(
            reconstruction_4[
                "mean_captured_energy"
            ]
        ),
    },
    {
        "analysis": "heldout_reconstruction",
        "metric": "8D captured energy",
        "value": float(
            reconstruction_8[
                "mean_captured_energy"
            ]
        ),
    },
    {
        "analysis": "heldout_reconstruction",
        "metric": "16D captured energy",
        "value": float(
            reconstruction_16[
                "mean_captured_energy"
            ]
        ),
    },
])


final_prediction_summary_df = pd.DataFrame([
    {
        "representation": "promoter direction",
        "dimensions": 1,
        "out_of_fold_r2": float(
            promoter_direction_result[
                "out_of_fold_r2"
            ]
        ),
        "pearson_r": float(
            promoter_direction_result[
                "pearson_r"
            ]
        ),
        "spearman_rho": float(
            promoter_direction_result[
                "spearman_rho"
            ]
        ),
    },
    {
        "representation": "SVD subspace",
        "dimensions": 1,
        "out_of_fold_r2": float(
            svd_1_result["out_of_fold_r2"]
        ),
        "pearson_r": float(
            svd_1_result["pearson_r"]
        ),
        "spearman_rho": float(
            svd_1_result["spearman_rho"]
        ),
    },
    {
        "representation": "SVD subspace",
        "dimensions": 4,
        "out_of_fold_r2": float(
            svd_4_result["out_of_fold_r2"]
        ),
        "pearson_r": float(
            svd_4_result["pearson_r"]
        ),
        "spearman_rho": float(
            svd_4_result["spearman_rho"]
        ),
    },
    {
        "representation": "SVD subspace",
        "dimensions": 8,
        "out_of_fold_r2": float(
            svd_8_result["out_of_fold_r2"]
        ),
        "pearson_r": float(
            svd_8_result["pearson_r"]
        ),
        "spearman_rho": float(
            svd_8_result["spearman_rho"]
        ),
    },
    {
        "representation": "SVD subspace",
        "dimensions": 16,
        "out_of_fold_r2": float(
            svd_16_result["out_of_fold_r2"]
        ),
        "pearson_r": float(
            svd_16_result["pearson_r"]
        ),
        "spearman_rho": float(
            svd_16_result["spearman_rho"]
        ),
    },
    {
        "representation": "SVD subspace",
        "dimensions": 64,
        "out_of_fold_r2": float(
            svd_64_result["out_of_fold_r2"]
        ),
        "pearson_r": float(
            svd_64_result["pearson_r"]
        ),
        "spearman_rho": float(
            svd_64_result["spearman_rho"]
        ),
    },
])


print("Final low-rank subspace summary:")
display(final_subspace_summary_df)

print("\nFinal downstream prediction summary:")
display(final_prediction_summary_df)


# ------------------------------------------------------------
# Machine-readable summary
# ------------------------------------------------------------

notebook7_summary = {
    "notebook": "07_promoter_mlp_low_rank_subspace",
    "model": "quietflamingo/dnabert2-no-flashattention",
    "representation": (
        "Mean-pooled Layer-1 MLP output"
    ),
    "primary_intervention_vector": (
        "GC-increased minus GC-decreased "
        "Layer-1 MLP representation"
    ),
    "sample_structure": {
        "biological_sources": 378,
        "source_dose_vectors": 756,
        "mutation_levels": [4, 8],
        "seeds_averaged_per_source_dose": 5,
        "representation_dimension": 768,
    },
    "pooled_uncentered_svd": {
        "pc1_energy_fraction": float(
            primary_svd_row[
                "energy_top_1"
            ]
        ),
        "participation_ratio": float(
            primary_svd_row[
                "participation_ratio"
            ]
        ),
        "pc1_promoter_direction_alignment": float(
            primary_svd_row[
                "top_component_alignment_with_promoter_direction"
            ]
        ),
        "rank_for_80_percent_energy": int(
            primary_svd_row[
                "rank_for_80_percent"
            ]
        ),
        "rank_for_90_percent_energy": int(
            primary_svd_row[
                "rank_for_90_percent"
            ]
        ),
    },
    "heldout_reconstruction": {
        "pc1_mean_energy_fraction": float(
            reconstruction_1[
                "mean_captured_energy"
            ]
        ),
        "pc1_promoter_projection_retained": float(
            reconstruction_1[
                "mean_promoter_projection_retained"
            ]
        ),
        "four_dimensional_energy_fraction": float(
            reconstruction_4[
                "mean_captured_energy"
            ]
        ),
        "eight_dimensional_energy_fraction": float(
            reconstruction_8[
                "mean_captured_energy"
            ]
        ),
        "sixteen_dimensional_energy_fraction": float(
            reconstruction_16[
                "mean_captured_energy"
            ]
        ),
    },
    "downstream_prediction": {
        "promoter_direction_r2": float(
            promoter_direction_result[
                "out_of_fold_r2"
            ]
        ),
        "svd_1_r2": float(
            svd_1_result[
                "out_of_fold_r2"
            ]
        ),
        "svd_4_r2": float(
            svd_4_result[
                "out_of_fold_r2"
            ]
        ),
        "svd_8_r2": float(
            svd_8_result[
                "out_of_fold_r2"
            ]
        ),
        "svd_16_r2": float(
            svd_16_result[
                "out_of_fold_r2"
            ]
        ),
        "svd_64_r2": float(
            svd_64_result[
                "out_of_fold_r2"
            ]
        ),
    },
    "main_supported_claim": (
        "The GC-responsive Layer-1 MLP change contains "
        "a dominant low-dimensional direction strongly "
        "aligned with promoter representation. This "
        "direction generalizes to held-out sequences and "
        "preserves nearly all of the mean promoter-aligned "
        "GC shift. A small set of low-rank coordinates "
        "predicts a substantial portion of the downstream "
        "Layer-5 diagnostic-probe effect."
    ),
    "claim_boundary": [
        (
            "Low-rank prediction and alignment do not "
            "establish causal mediation through the "
            "identified subspace."
        ),
        (
            "The downstream target is a fixed diagnostic "
            "probe, not a native promoter-classification "
            "output of DNABERT-2."
        ),
        (
            "The dominant shared GC direction is low-rank, "
            "but centered sequence-specific residuals are "
            "substantially more distributed."
        ),
        (
            "GC-changing substitutions also alter local "
            "sequence context, motifs, dinucleotides, and "
            "tokenization."
        ),
    ],
}


with open(
    NOTEBOOK7_DIR
    / "notebook7_summary.json",
    "w",
    encoding="utf-8",
) as output_file:
    json.dump(
        notebook7_summary,
        output_file,
        indent=2,
    )


# ------------------------------------------------------------
# Human-readable results file
# ------------------------------------------------------------

summary_markdown = f"""
# Notebook 7 — Layer-1 MLP Low-Rank Subspace

## Main result

The GC-increase versus GC-decrease change in DNABERT-2's
Layer-1 MLP output contains a dominant low-dimensional
direction.

In the pooled uncentered SVD:

- PC1 explained **{primary_svd_row["energy_top_1"]:.3f}**
  of total intervention-vector energy.
- The participation ratio was
  **{primary_svd_row["participation_ratio"]:.2f}**.
- PC1 aligned with the Layer-1 promoter direction at
  **{primary_svd_row["top_component_alignment_with_promoter_direction"]:.3f}**.

## Held-out generalization

When the subspace was learned without each held-out
biological source:

- PC1 captured **{reconstruction_1["mean_captured_energy"]:.3f}**
  of intervention-vector energy.
- PC1 retained **{reconstruction_1["mean_promoter_projection_retained"]:.3f}**
  of the mean promoter-direction shift.
- Four dimensions captured
  **{reconstruction_4["mean_captured_energy"]:.3f}**.
- Eight dimensions captured
  **{reconstruction_8["mean_captured_energy"]:.3f}**.
- Sixteen dimensions captured
  **{reconstruction_16["mean_captured_energy"]:.3f}**.

## Downstream association

Held-out prediction of the Layer-5 diagnostic-probe effect:

- promoter-direction scalar:
  **R² = {promoter_direction_result["out_of_fold_r2"]:.3f}**
- one SVD coordinate:
  **R² = {svd_1_result["out_of_fold_r2"]:.3f}**
- four coordinates:
  **R² = {svd_4_result["out_of_fold_r2"]:.3f}**
- eight coordinates:
  **R² = {svd_8_result["out_of_fold_r2"]:.3f}**
- sixteen coordinates:
  **R² = {svd_16_result["out_of_fold_r2"]:.3f}**
- sixty-four coordinates:
  **R² = {svd_64_result["out_of_fold_r2"]:.3f}**

Most predictable downstream variation is captured by the
first few dimensions, with diminishing improvement at higher
rank.

## Interpretation

DNABERT-2's Layer-1 MLP contains a dominant shared
GC-responsive direction closely aligned with promoter
representation. Sequence-specific residual responses remain
more distributed.

## Claim boundary

This analysis establishes low-dimensional structure,
held-out generalization, and downstream association. It does
not establish that this subspace causally mediates the full
Layer-5 probe effect.
""".strip()


with open(
    NOTEBOOK7_DIR
    / "NOTEBOOK7_RESULTS.md",
    "w",
    encoding="utf-8",
) as output_file:
    output_file.write(
        summary_markdown
    )


final_subspace_summary_df.to_csv(
    NOTEBOOK7_DIR
    / "notebook7_final_subspace_summary.csv",
    index=False,
)

final_prediction_summary_df.to_csv(
    NOTEBOOK7_DIR
    / "notebook7_final_prediction_summary.csv",
    index=False,
)


print("\nSaved final Notebook 7 files:")

for filename in [
    "notebook7_final_subspace_summary.csv",
    "notebook7_final_prediction_summary.csv",
    "notebook7_summary.json",
    "NOTEBOOK7_RESULTS.md",
]:
    print(
        NOTEBOOK7_DIR / filename
    )

print(
    "\nNotebook 7 completed successfully."
)

Final low-rank subspace summary:


,analysis,metric,value
0,pooled_uncentered_svd,PC1 energy fraction,0.570767
1,pooled_uncentered_svd,participation ratio,2.988808
2,pooled_uncentered_svd,PC1 alignment with promoter direction,0.894321
3,heldout_reconstruction,PC1 captured energy,0.446160
4,heldout_reconstruction,PC1 promoter projection retained,0.982344
5,heldout_reconstruction,4D captured energy,0.590516
6,heldout_reconstruction,8D captured energy,0.640865
7,heldout_reconstruction,16D captured energy,0.690300



Final downstream prediction summary:


,representation,dimensions,out_of_fold_r2,pearson_r,spearman_rho
0,promoter direction,1,0.549133,0.741040,0.720624
1,SVD subspace,1,0.554241,0.744475,0.706500
2,SVD subspace,4,0.602831,0.776436,0.749488
3,SVD subspace,8,0.611631,0.782096,0.759497
4,SVD subspace,16,0.615998,0.784956,0.760119
5,SVD subspace,64,0.623145,0.790919,0.754726



Saved final Notebook 7 files:
/content/drive/MyDrive/algoverse/dnabert/outputs/promoter_mlp_low_rank_subspace/notebook7_final_subspace_summary.csv
/content/drive/MyDrive/algoverse/dnabert/outputs/promoter_mlp_low_rank_subspace/notebook7_final_prediction_summary.csv
/content/drive/MyDrive/algoverse/dnabert/outputs/promoter_mlp_low_rank_subspace/notebook7_summary.json
/content/drive/MyDrive/algoverse/dnabert/outputs/promoter_mlp_low_rank_subspace/NOTEBOOK7_RESULTS.md

Notebook 7 completed successfully.
